In [ ]:
import pandas as pd
import pycountry
import json
from os import listdir as ls
from datetime import timedelta

from emu_renewal.inputs import DATA_PATH, get_apple_mobility, get_worldbank_national_pop, \
    get_country_vacc_data, get_indicator_series_from_who_data, get_country_vacc_data
from emu_renewal.run import find_run_start_time, find_run_end_time

pd.options.plotting.backend = "plotly"

In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
pop_threshold = 1e6
gmob_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "gmob" in i]
fbmob_avail = [i[:3] for i in ls(DATA_PATH / "mobility") if "fbmob" in i]
amob_avail = []
amob_unavail = []
for c in pycountry.countries:
    try:
        iso3 = c.alpha_3
        get_apple_mobility(iso3)
        amob_avail.append(iso3)
    except:
        amob_unavail.append(iso3)
any_mob_avail = list(set(amob_avail + fbmob_avail + gmob_avail))

In [ ]:
mob_6months_avail = []
cases_data = pd.DataFrame(columns=mob_6months_avail)
for c in any_mob_avail:
    deaths_start_threshold = 2e-6
    most_extreme_prop = 0.05
    vacc_data = get_country_vacc_data(iso3)
    end_time = find_run_end_time(vacc_data, most_extreme_prop)
    try:
        pop = get_worldbank_national_pop(iso3)
    except:
        pop = get_undesa_national_pop(iso3)
    deaths_data = get_indicator_series_from_who_data("New_deaths", c)
    data_start = find_run_start_time(deaths_data, pop, deaths_start_threshold)
    duration = end_time - data_start
    if duration > timedelta(180):
        mob_6months_avail.append(c)
    cases_data[c] = get_indicator_series_from_who_data("New_cases", c)

In [ ]:
cases_data.shape

## Sorted
- Vaccination
- Population
## Issues
### Singapore
Hospitalisation data only available from OWID from 26/2/2023
CoVariant numbers very low
### South Korea
Only hospitalisation data are ICU occupancy
### United States
Something going on with variants in early stages - need to look into further - may not be enough data from CoVariants
### Israel
Sorted, but Facebook mobility data unavailable
### Chile
Check on variants
### Malaysia
Low numbers in early phases - small window to run analysis by standard criteria
Variant issues too
### Serbia
Occupancy for hospitalisations
Variant issues
### Iceland
No Google mobility data
Sparse time series data - deaths look unreliable
### Liechtenstein
Most mobility unavailable
Sparse time series data - deaths look unreliable